<a href="https://colab.research.google.com/github/beenishyousaf94-sketch/UlsterGPT/blob/main/UlsterGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install / upgrade the ML packages
!pip install -q -U \
  transformers \
  peft \
  accelerate \
  bitsandbytes \
  datasets \
  sentence-transformers \
  faiss-cpu \
  rouge-score \
  trl \
  gradio \
  evaluate \
  nltk \
  "pandas==2.2.3" \
  "pyarrow<24" \
  "requests==2.32.4"

print("✅ Install done. Now click Runtime → Restart session, then run the next cell.")

✅ Install done. Now click Runtime → Restart session, then run the next cell.


In [ ]:
import os
import re
import json
import faiss
import torch
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin, urlparse

from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from sklearn.metrics import precision_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

print("✅ All modules imported successfully!")
print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")

✅ All modules imported successfully!
✅ GPU detected: NVIDIA A100-SXM4-80GB


In [ ]:
MIN_CHARS = 400
MAX_CHARS = 1200

pages_df = pd.read_csv("ulster_fees_data.csv", encoding="utf-8-sig")
pages_df = pages_df.dropna(subset=["chunk_text"]).reset_index(drop=True)

print("✅ Loaded chunks:", len(pages_df))
print("Pages:", pages_df["url"].nunique())
pages_df.head()

✅ Loaded chunks: 3032
Pages: 197


,url,page_title,chunk_text
0,https://www.ulster.ac.uk/global/apply/fees-and...,Fees and Finance for International Students | ...,Fees and Finance for International Students | ...
1,https://www.ulster.ac.uk/global/apply/fees-and...,Fees and Finance for International Students | ...,Guide to Ulster Staff – search and browse Ulst...
2,https://www.ulster.ac.uk/global/apply/fees-and...,Fees and Finance for International Students | ...,University For International Students Fees and...
3,https://www.ulster.ac.uk/global/apply/fees-and...,Fees and Finance for International Students | ...,Scholarships Accommodation Your Country Studen...
4,https://www.ulster.ac.uk/global/apply/fees-and...,Fees and Finance for International Students | ...,Team at Ulster Global Partnerships Global Budd...


In [ ]:
SRC = "Ulster University Tuition Fees Payment Policy 2026/27 and fee pages"
manual_queries = []

def add(questions, answer, ctx=SRC):
    for q in questions:
        manual_queries.append({"instruction": q, "input": ctx, "output": answer})

# --- Instalments (new September 2026 starters) ---
countries = ["Pakistan", "India", "Nigeria", "China", "Egypt", "Yemen", "Sudan", "Iran", "Bangladesh", "Ghana"]
for c in countries:
    add([f"I am an international student from {c}. Can I pay my tuition fees in instalments?",
         f"Is there a payment plan for students from {c}?"],
        "Yes. If you cannot pay in full, you must pay at least 50% of the tuition fee (including your deposit) "
        "at registration. The remaining balance is paid in two equal instalments on 01 December and 01 March "
        "via the International Payment Portal.")

add(["When are the instalment dates for international students starting in September 2026?",
     "What are the payment deadlines for new international students?"],
    "New international students starting in September 2026 pay at least 50% (including the deposit) at "
    "registration, then two equal instalments on 01 December and 01 March.")

add(["When are the instalments due if I start a postgraduate course in January 2027?",
     "What is the payment plan for January 2027 postgraduate entry?"],
    "Full-time postgraduate students starting in January 2027 pay at least 50% of the programme fee "
    "(including the deposit) before registration, then two equal instalments on 01 April and 01 July.")

add(["I am a returning international student. How do instalments work for year 2?",
     "What is the instalment plan for re-enrolling international students?"],
    "Returning international students must pay at least 25% of the year's fee before registration, "
    "then 25% on 01 December and the remaining 50% on 01 March.")

add(["Can international MBBS students pay in instalments?",
     "What is the payment plan for international medicine students?"],
    "Yes. International MBBS students who cannot pay in full must pay at least 10% of the tuition fee as a "
    "deposit and pay the balance over 9 equal instalments, signing up at registration. A clinical placement "
    "levy (£12,133 in 2026/27) is also payable each year.")

# --- Fees 2026/27 ---
fees = [
    ("an undergraduate degree", "£17,490"),
    ("a taught Masters course such as MSc Computer Science", "£18,310"),
    ("Postgraduate Research (PhD)", "£19,040"),
    ("the Executive MBA", "£22,680"),
    ("MBBS Medicine", "£39,630 (plus the clinical placement levy of £12,133)"),
    ("MSc Management and Corporate Governance", "£20,490"),
    ("a placement year on a sandwich course", "£4,900"),
]
for course, fee in fees:
    add([f"What is the 2026/27 international tuition fee for {course}?",
         f"How much does an international student pay per year for {course}?"],
        f"For 2026/27, the international tuition fee for {course} is {fee} per year. Fees are subject to an annual increase.")

# --- Deposit ---
add(["Do international students need to pay a deposit?",
     "Is the deposit refundable?",
     "Why do I have to pay a deposit before getting my CAS?"],
    "Yes. International students studying on campus must pay a deposit before a Confirmation of Acceptance "
    "for Studies (CAS) can be issued. It is non-refundable, except in cases such as a visa refusal (not for "
    "fraud), no CAS being issued, the University being unable to offer a place, the programme being cancelled, "
    "or approved personal circumstances.")

# --- Payment methods ---
add(["How can international students pay their tuition fees?",
     "Can I pay my tuition fees in cash?",
     "What payment methods does Ulster accept from overseas?"],
    "International students can pay online by debit/credit card or through Flywire, which allows payment by "
    "bank transfer, card or e-wallet in your local currency. Cash is not accepted for tuition fees.")

# --- Sanctions ---
add(["What happens if I don't pay my tuition fees on time?",
     "What are the sanctions for non-payment of fees?"],
    "Sanctions include: termination of your payment plan; restricted access to Blackboard, IT and library "
    "facilities; being unable to submit assessments or receive marks; being prevented from re-enrolling or "
    "graduating; and possible withdrawal of your Student Visa sponsorship.")

print(f"✅ {len(manual_queries)} verified Q&As")

✅ 50 verified Q&As


In [ ]:
import random

COUNTRIES = ["pakistan", "india", "nigeria", "brazil", "egypt", "uae", "china",
             "yemen", "sudan", "iran", "syria", "afghanistan", "bangladesh", "ghana"]

INSTALMENT_ANSWER = ("Yes. If you cannot pay in full, you must pay at least 50% of the tuition fee "
                     "(including your deposit) at registration. The remaining balance is paid in two equal "
                     "instalments on 01 December and 01 March via the International Payment Portal.")

def find_sentences_with(patterns, text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if any(p in s.lower() for p in patterns)]

def build_instruction_data(df):
    data = []
    for _, row in df.iterrows():
        text, url, title = row["chunk_text"], row["url"], row["page_title"]
        low = text.lower()

        # A) Instalment questions answered with real page text
        inst = find_sentences_with(["instalment", "installment"], text)
        if inst:
            joined = " ".join(inst)
            data.append({
                "instruction": "Are students allowed to pay their tuition fees in instalments at Ulster University?",
                "input": f"PAGE TITLE: {title}\nSOURCE: {url}\nRELEVANT TEXT:\n{joined}",
                "output": joined
            })

        # B) Country questions: same policy for every country (no high-risk rule exists)
        country_sents = find_sentences_with(COUNTRIES, text)
        if country_sents:
            ctx = f"PAGE TITLE: {title}\nSOURCE: {url}\nRELEVANT TEXT:\n" + " ".join(country_sents)
            for c in COUNTRIES:
                if c in low:
                    data.append({
                        "instruction": f"I am a student from {c.capitalize()}. Can I pay my tuition fees in instalments?",
                        "input": ctx, "output": INSTALMENT_ANSWER
                    })

        # C) Real fee statements (£ lines)
        for line in text.split("."):
            if "£" in line and len(line.strip()) > 20:
                fee_line = line.strip()
                data.append({
                    "instruction": "What does Ulster University say about this fee?",
                    "input": f"PAGE TITLE: {title}\nSOURCE: {url}\nFEE LINE:\n{fee_line}",
                    "output": fee_line + "."
                })
    return data

scraped = build_instruction_data(pages_df)

random.seed(42)
random.shuffle(scraped)
scraped = scraped[:4000]

instruction_data = scraped + manual_queries * 10   # verified Q&As repeated 10x
random.shuffle(instruction_data)

with open("ulster_instructions.json", "w") as f:
    json.dump(instruction_data, f, indent=2)

print(f"Scraped examples used: {len(scraped)}")
print(f"Manual examples (x10): {len(manual_queries) * 10}")
print(f"✅ Total for training:  {len(instruction_data)}")

Scraped examples used: 4000
Manual examples (x10): 500
✅ Total for training:  4500


In [ ]:
import gc

# ---- 1. Clear the failed run from GPU memory ----
for name in ["trainer", "model", "train_dataset", "dataset"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print("🧹 GPU memory after cleanup:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

os.environ["WANDB_DISABLED"] = "true"

# ---- 2. Load and format the training data ----
dataset = load_dataset("json", data_files="ulster_instructions.json")["train"]

SYSTEM = ("You are UlsterGPT, an expert on Ulster University tuition fees and admission policies. "
          "Answer the query ONLY using the given context.")

def formatting_func(examples):
    texts = []
    for instr, ctx, resp in zip(examples["instruction"], examples["input"], examples["output"]):
        # No <s> or </s>: the tokenizer adds BOS and the trainer adds EOS
        texts.append(f"[INST] {SYSTEM}\n\nContext:\n{ctx}\n\nQuery: {instr} [/INST] {resp}")
    return {"text": texts}

train_dataset = dataset.map(formatting_func, batched=True, remove_columns=dataset.column_names)
print("✅ Train size:", len(train_dataset))

# ---- 3. Load a fresh Mistral-7B in 4-bit ----
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
model.config.use_cache = False

print("✅ Fresh model loaded. GPU memory used:",
      round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

🧹 GPU memory after cleanup: 5.0 GB


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

✅ Train size: 4500


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Fresh model loaded. GPU memory used: 9.1 GB


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="./ulstergpt_results",
    dataset_text_field="text",
    max_length=1024,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,      # off: avoids the CheckpointError (A100 has enough memory)
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.model.print_trainable_parameters()
trainer.train()

# Save only the LoRA adapter
trainer.model.save_pretrained("./ulstergpt_adapter")
tokenizer.save_pretrained("./ulstergpt_adapter")
print("✅ Training done, adapter saved to ./ulstergpt_adapter")

Adding EOS to train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

trainable params: 83,886,080 || all params: 7,325,618,176 || trainable%: 1.1451


Step,Training Loss
25,0.978672
50,0.139148
75,0.072070
100,0.061796
125,0.058887
150,0.048528
175,0.067516
200,0.065932
225,0.052343
250,0.057331


✅ Training done, adapter saved to ./ulstergpt_adapter


In [ ]:
# ---- 1. Knowledge base = scraped pages + verified 2026/27 facts ----
kb = pages_df[["url", "page_title", "chunk_text"]].copy()

# Add each verified answer once, so retrieval always finds the official facts
verified = {}
for q in manual_queries:
    verified.setdefault(q["output"], q["instruction"])
verified_df = pd.DataFrame([
    {"url": "Tuition Fees Payment Policy 2026/27", "page_title": "VERIFIED: " + instr, "chunk_text": ans}
    for ans, instr in verified.items()
])
kb = pd.concat([verified_df, kb], ignore_index=True).dropna(subset=["chunk_text"])
print(f"Knowledge base: {len(verified_df)} verified facts + {len(pages_df)} scraped chunks = {len(kb)}")

# ---- 2. Embed with a stronger retrieval model ----
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")

docs = (kb["page_title"].fillna("") + ". " + kb["chunk_text"]).tolist()
chunk_embeddings = embed_model.encode(
    docs, batch_size=128, show_progress_bar=True, normalize_embeddings=True
).astype("float32")

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
chunk_meta = kb.to_dict(orient="records")
print("✅ FAISS index built:", index.ntotal, "vectors")

# ---- 3. Retrieval function ----
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve_context(query, k=3, return_sources=False):
    q_emb = embed_model.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    hits = [chunk_meta[i] for i in ids[0]]
    context = "\n\n".join(h["chunk_text"] for h in hits)
    if return_sources:
        return context, [(h["url"], round(float(s), 3)) for h, s in zip(hits, scores[0])]
    return context

# ---- 4. Quick check ----
for q in ["Can a student from Yemen pay in instalments?",
          "What is the fee for a Masters in 2026/27?",
          "Can I pay in cash?"]:
    ctx, src = retrieve_context(q, return_sources=True)
    print(f"\n🔎 {q}\n   top source: {src[0]}\n   {ctx[:200]}...")

Knowledge base: 15 verified facts + 3032 scraped chunks = 3047


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

✅ FAISS index built: 3047 vectors

🔎 Can a student from Yemen pay in instalments?
   top source: ('Tuition Fees Payment Policy 2026/27', 0.701)
   Yes. If you cannot pay in full, you must pay at least 50% of the tuition fee (including your deposit) at registration. The remaining balance is paid in two equal instalments on 01 December and 01 Marc...

🔎 What is the fee for a Masters in 2026/27?
   top source: ('Tuition Fees Payment Policy 2026/27', 0.779)
   For 2026/27, the international tuition fee for a taught Masters course such as MSc Computer Science is £18,310 per year. Fees are subject to an annual increase.

For 2026/27, the international tuition...

🔎 Can I pay in cash?
   top source: ('Tuition Fees Payment Policy 2026/27', 0.644)
   International students can pay online by debit/credit card or through Flywire, which allows payment by bank transfer, card or e-wallet in your local currency. Cash is not accepted for tuition fees.

F...


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/UlsterGPT"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. LoRA adapter + tokenizer (the trained model)
trainer.model.save_pretrained(f"{SAVE_DIR}/ulstergpt_adapter")
tokenizer.save_pretrained(f"{SAVE_DIR}/ulstergpt_adapter")

# 2. Data files
for f in ["ulster_fees_data.csv", "ulster_instructions.json"]:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)

# 3. RAG knowledge base + FAISS index
kb.to_csv(f"{SAVE_DIR}/knowledge_base.csv", index=False)
faiss.write_index(index, f"{SAVE_DIR}/ulster_faiss.index")

# 4. Training log (loss per step) for your paper
pd.DataFrame(trainer.state.log_history).to_csv(f"{SAVE_DIR}/training_log.csv", index=False)

print("✅ Saved to Google Drive:")
for f in sorted(os.listdir(SAVE_DIR)):
    print("   ", f)

Mounted at /content/drive
✅ Saved to Google Drive:
    knowledge_base.csv
    training_log.csv
    ulster_faiss.index
    ulster_fees_data.csv
    ulster_instructions.json
    ulstergpt_adapter


In [ ]:
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True
gen_tokenizer = tokenizer

SYSTEM = ("You are UlsterGPT, an expert on Ulster University tuition fees and admission policies. "
          "Answer the query ONLY using the given context. Answer only what the student asked. "
          "Do not mention MBBS, medicine or other courses unless the question is about them.")

MED_WORDS = ["mbbs", "medicine", "medical", "doctor"]

# Fill this ONLY if you have an official, citable source (e.g. offer letter / Fees Team email)
RESTRICTED_COUNTRIES = []          # e.g. ["yemen", "sudan"]
RESTRICTED_SOURCE = ""             # e.g. "Ulster offer letter, 2026"

def retrieve_context(query, k=3, return_sources=False):
    q_emb = embed_model.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, 15)                 # search wider, then filter
    is_medical = any(w in query.lower() for w in MED_WORDS)
    hits = []
    for i, s in zip(ids[0], scores[0]):
        h = chunk_meta[i]
        text = str(h["chunk_text"]).lower()
        if not is_medical and ("mbbs" in text or "clinical placement levy" in text):
            continue                                       # drop medicine-only rules
        hits.append((h, float(s)))
        if len(hits) == k:
            break
    context = "\n\n".join(h["chunk_text"] for h, _ in hits)
    if return_sources:
        return context, [(h["url"], round(s, 3)) for h, s in hits]
    return context

def ulstergpt_answer(query, max_new_tokens=200, k=3, show_sources=False, model=None):
    model = model or ft_model
    ql = query.lower()

    # Optional rule, only active if you add a real source above
    if RESTRICTED_COUNTRIES and any(c in ql for c in RESTRICTED_COUNTRIES) and "instal" in ql:
        return (f"No. Students from this country must pay tuition fees in full at or before enrolment. "
                f"(Source: {RESTRICTED_SOURCE})")

    context, sources = retrieve_context(query, k=k, return_sources=True)
    prompt = f"[INST] {SYSTEM}\n\nContext:\n{context}\n\nQuery: {query} [/INST]"
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    answer = gen_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if show_sources:
        answer += "\n\n📚 Sources:\n" + "\n".join(f"- {u}" for u, _ in sources)
    return answer

# Test
for q in ["I am from Yemen. Can I pay in instalments?",
          "How much is MSc Computer Science for international students?",
          "Can I pay my fees in cash?",
          "Can MBBS students pay in instalments?"]:
    print("Q:", q)
    print("A:", ulstergpt_answer(q, show_sources=True), "\n")

Q: I am from Yemen. Can I pay in instalments?
A: Yes. If you cannot pay in full, you must pay at least 50% of the tuition fee (including your deposit) at registration. The remaining balance is paid in two equal instalments on 01 December and 01 March via the International Payment Portal. Returning international students must pay at least 25% of the year's fee before registration, then 25% on 01 December and the remaining 50% on 01 March. New international students starting in September 2026 pay at least 50% (including the deposit) at registration, then two equal instalments on 01 December and 01 March.

📚 Sources:
- Tuition Fees Payment Policy 2026/27
- Tuition Fees Payment Policy 2026/27
- Tuition Fees Payment Policy 2026/27 

Q: How much is MSc Computer Science for international students?
A: For 2026/27, the international tuition fee for a taught Masters course such as MSc Computer Science is £18,310 per year. Fees are subject to an annual increase.

📚 Sources:
- Tuition Fees Payment

In [ ]:
import gradio as gr

def chat_interface(user_query):
    if not user_query or not user_query.strip():
        return "Please enter a question."
    try:
        return ulstergpt_answer(user_query, show_sources=True)
    except Exception as e:
        return f"⚠️ Error: {e}"

examples = [
    ["I am from Pakistan. Can I pay my tuition in instalments?"],
    ["What is the 2026/27 fee for an international Masters?"],
    ["When are instalments due if I start in January 2027?"],
    ["I'm a returning student. How do instalments work?"],
    ["Is the deposit refundable if my visa is refused?"],
    ["How can I pay my fees from abroad?"],
    ["What happens if I miss a payment?"],
    ["Can MBBS students pay in instalments?"],
]

with gr.Blocks(title="UlsterGPT") as demo:
    gr.Markdown("# 🎓 UlsterGPT: Tuition & Fees Assistant")
    gr.Markdown(
        "Answers are based on Ulster University's **2026/27** fee pages and Tuition Fees Payment Policy. "
        "Always confirm with the Fees Team before making payments."
    )
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Your question", lines=2,
                placeholder="e.g. I am from Nigeria, can I pay in instalments?"
            )
            with gr.Row():
                submit_btn = gr.Button("Get Answer", variant="primary")
                clear_btn = gr.Button("Clear")
        with gr.Column():
            output_text = gr.Textbox(label="UlsterGPT", lines=10, interactive=False)

    gr.Examples(examples=examples, inputs=input_text, label="Try these questions")

    submit_btn.click(chat_interface, inputs=input_text, outputs=output_text)
    input_text.submit(chat_interface, inputs=input_text, outputs=output_text)
    clear_btn.click(lambda: ("", ""), outputs=[input_text, output_text])

# Gradio 6 takes the theme in launch(); older versions don't
try:
    demo.launch(share=True, show_error=True, theme=gr.themes.Soft())
except TypeError:
    demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://989889faade668e5a4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Held-out test set: (question, required key facts, reference answer)
TEST = [
    ("My daughter is coming from Pakistan in September. Can she spread the fee payments?",
     ["50%", "december", "march"], "Yes. Pay at least 50% at registration, then two equal instalments on 01 December and 01 March."),
    ("Do Sudanese students have to pay the whole fee upfront?",
     ["50%", "december", "march"], "No. They can pay at least 50% at registration and the rest on 01 December and 01 March."),
    ("I'm Iranian, is a payment plan possible for me?",
     ["50%", "december", "march"], "Yes. At least 50% at registration, then instalments on 01 December and 01 March."),
    ("How much does a taught postgraduate degree cost overseas students this year?",
     ["18,310"], "The 2026/27 international fee for a taught Masters is £18,310 per year."),
    ("What's the yearly cost of a bachelor's degree for international students in 2026/27?",
     ["17,490"], "The 2026/27 international undergraduate fee is £17,490 per year."),
    ("What are the PhD fees for international students?",
     ["19,040"], "Postgraduate Research costs £19,040 per year for international students in 2026/27."),
    ("How much is medicine for an international student?",
     ["39,630"], "MBBS costs £39,630 per year plus the clinical placement levy."),
    ("Is there an extra charge for international medical students?",
     ["12,133"], "Yes, a clinical placement levy of £12,133 in 2026/27, payable each year."),
    ("What does the Executive MBA cost?",
     ["22,680"], "The Executive MBA costs £22,680 for international students in 2026/27."),
    ("What do I pay for my sandwich placement year?",
     ["4,900"], "The placement year fee is £4,900."),
    ("I start my MSc in January 2027. When are my instalments?",
     ["april", "july"], "At least 50% before registration, then equal instalments on 01 April and 01 July."),
    ("I'm going into second year. How much must I pay before re-enrolling?",
     ["25%"], "At least 25% before registration, then 25% on 01 December and 50% on 01 March."),
    ("Will I get my deposit back if UKVI refuses my visa?",
     ["visa", "refund"], "The deposit is non-refundable except in cases such as a visa refusal not due to fraud."),
    ("Can I bring cash to pay my tuition?",
     ["cash", "not"], "No, cash is not accepted for tuition fees."),
    ("Which service should I use to pay from abroad in my own currency?",
     ["flywire"], "Use Flywire, which supports bank transfer, card or e-wallet in local currency."),
    ("Could not paying my fees affect my student visa?",
     ["sponsorship"], "Yes, your Student Visa sponsorship may be withdrawn."),
    ("Will I lose library access if I miss a payment?",
     ["library"], "Yes, access to Blackboard, IT and library facilities will be restricted."),
    ("Do I have to pay anything before I get my CAS?",
     ["deposit", "cas"], "Yes, a deposit must be paid before a CAS can be issued."),
    ("How do instalments work for international MBBS students?",
     ["10%", "9"], "Pay at least 10% as a deposit, then the balance in 9 equal instalments."),
    ("If I can't pay everything, what's the minimum due at registration as a new student?",
     ["50%"], "At least 50% of the tuition fee, including the deposit."),
]

def norm(t):
    return re.sub(r"\s+", " ", (t or "").lower().replace("£", "").replace(",", ""))

def has_facts(answer, keys):
    a = norm(answer)
    return all(norm(k) in a or (k == "refund" and "refundable" in a) for k in keys)

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1

def evaluate(name):
    rows = []
    for q, keys, ref in tqdm(TEST, desc=name):
        pred = ulstergpt_answer(q)
        r = scorer.score(ref.lower(), pred.lower())
        rows.append({
            "system": name, "question": q, "prediction": pred,
            "correct": has_facts(pred, keys),
            "rouge1": r["rouge1"].fmeasure, "rougeL": r["rougeL"].fmeasure,
            "bleu": sentence_bleu([ref.lower().split()], pred.lower().split(), smoothing_function=smooth),
        })
    return pd.DataFrame(rows)

# 1) Your fine-tuned model + RAG
res_ft = evaluate("UlsterGPT (QLoRA + RAG)")

# 2) Base Mistral + same RAG (LoRA adapter switched off)
with ft_model.disable_adapter():
    res_base = evaluate("Base Mistral + RAG")

results = pd.concat([res_base, res_ft], ignore_index=True)
summary = (results.groupby("system")[["correct", "rouge1", "rougeL", "bleu"]]
           .mean().round(3).rename(columns={"correct": "Factual Accuracy"}))

results.to_csv(f"{SAVE_DIR}/evaluation_results.csv", index=False)
summary.to_csv(f"{SAVE_DIR}/evaluation_summary.csv")

print("=== Held-out evaluation (20 unseen questions) ===")
display(summary)

print("\n❌ Questions UlsterGPT got wrong:")
wrong = res_ft[~res_ft["correct"]]
if wrong.empty:
    print("None 🎉")
for _, r in wrong.iterrows():
    print(f"- {r['question']}\n  → {r['prediction'][:250]}\n")

Base Mistral + RAG: 100%|██████████| 20/20 [01:33<00:00,  4.69s/it]

=== Held-out evaluation (20 unseen questions) ===


,Factual Accuracy,rouge1,rougeL,bleu
system,,,,
Base Mistral + RAG,0.9,0.342,0.292,0.063
UlsterGPT (QLoRA + RAG),0.9,0.380,0.330,0.099



❌ Questions UlsterGPT got wrong:
- I'm Iranian, is a payment plan possible for me?
  → Yes. If you cannot pay in full, you must pay at least 50% of the tuition fee and agree to pay the balance in instalments. Instalment plan to cover the balance is two further equal instalments on 01 April and 01 July via the International Payment Port

- Will I lose library access if I miss a payment?
  → Yes. If you cannot pay your tuition fees in full, you must pay at least 50% of the tuition fee (including your deposit) at registration. The remaining balance is paid in two equal instalments on 01 December and 01 March via the International Payment 



In [ ]:
# =========================================================
# PILOT: Table vs Text representation for LLM policy QA
# Same verified Ulster 2026/27 facts, two representations,
# same questions, same model (Mistral-7B-Instruct, no fine-tuning),
# greedy decoding. Scored by required key facts + forbidden facts.
# ============================================================
import re, json, time
import pandas as pd

TEXT_REP = """International tuition fees for 2026/27 are as follows. Undergraduate and integrated masters courses cost £17,490 per year. A placement year or intercalary year abroad costs £4,900. Taught Masters courses cost £18,310 per year, except MSc Management and Corporate Governance, MSc Biotechnology Research, MSc Human Nutrition, MSc Pharmaceutical Sciences, LLM Access to Justice and LLM International Commercial Law and ADR, which cost £20,490. Postgraduate Research costs £19,040 per year. The Executive MBA and MPA cost £22,680. MBBS Medicine costs £39,630 per year, and international medical students also pay a clinical placement levy of £12,133 each year on top of tuition.

International students studying on campus must pay a deposit before a CAS can be issued. The CAS deposit does not apply to international distance learning students. Tuition fees should preferably be paid in full at or before enrolment. New international students starting in September 2026 who cannot pay in full must pay at least 50% of the tuition fee, including the deposit, at registration, and the balance in two equal instalments on 01 December and 01 March. Full-time postgraduate students starting in January 2027 must pay at least 50% of the programme fee, including the deposit, and the balance in two equal instalments on 01 April and 01 July. Returning international students must pay at least 25% of the year's fee before registration, then 25% on 01 December and the remaining 50% on 01 March. International MBBS students who cannot pay in full must pay at least 10% of the tuition fee as a deposit and the balance over 9 equal instalments. These rules apply to all international students regardless of nationality.

An instalment plan can only be set up if the balance is £100 or more; smaller balances must be paid in full. Cash is not accepted. International students can pay by card online or through Flywire in their local currency. American Express and Laser cards are not accepted, and card payments are not taken over the telephone."""

TABLE_REP = """Table 1: International tuition fees 2026/27
| Course | Fee per year |
|---|---|
| Undergraduate / integrated masters | £17,490 |
| Placement year / intercalary year abroad | £4,900 |
| Taught Masters (standard) | £18,310 |
| MSc Management and Corporate Governance, MSc Biotechnology Research, MSc Human Nutrition, MSc Pharmaceutical Sciences, LLM Access to Justice, LLM International Commercial Law and ADR | £20,490 |
| Postgraduate Research | £19,040 |
| Executive MBA / MPA | £22,680 |
| MBBS Medicine | £39,630 |
| MBBS clinical placement levy (each year, on top of tuition) | £12,133 |

Table 2: Instalment plans (all international students, regardless of nationality)
| Student group | Minimum first payment | Remaining instalments |
|---|---|---|
| New student, September 2026 start | 50% of fee (incl. deposit) at registration | 2 equal instalments: 01 December and 01 March |
| Full-time postgraduate, January 2027 start | 50% of programme fee (incl. deposit) | 2 equal instalments: 01 April and 01 July |
| Returning student (year 2 onwards) | 25% of year's fee before registration | 25% on 01 December, 50% on 01 March |
| MBBS student | 10% of fee as deposit | 9 equal instalments |

Table 3: Other payment rules
| Rule | Detail |
|---|---|
| CAS deposit | Required before a CAS is issued for on-campus students; not for distance learners |
| Preferred payment | In full at or before enrolment |
| Minimum balance for instalment plan | £100 (smaller balances paid in full) |
| Cash | Not accepted |
| Payment methods | Card online, or Flywire in local currency |
| Cards not accepted | American Express, Laser; no telephone card payments |"""

# (id, category, question, required keys (all), forbidden keys (none may appear))
QUESTIONS = [
    ("L1", "lookup", "What is the yearly international fee for a taught Masters such as MSc Computer Science?", ["18310"], []),
    ("L2", "lookup", "How much does MSc Pharmaceutical Sciences cost per year?", ["20490"], ["18310"]),
    ("L3", "lookup", "What is the international fee for Postgraduate Research?", ["19040"], []),
    ("L4", "lookup", "How much is the Executive MBA?", ["22680"], []),
    ("L5", "lookup", "What does a placement year cost?", ["4900"], []),
    ("L6", "lookup", "Is cash accepted for tuition fees?", ["not"], []),
    ("L7", "lookup", "Can I pay with American Express?", ["not"], []),
    ("L8", "lookup", "Do distance learning students pay the CAS deposit?", ["not"], []),
    ("C1", "conditional", "I am a new undergraduate starting in September 2026. When are my instalments due?", ["december", "march"], ["april", "july"]),
    ("C2", "conditional", "I start a full-time MSc in January 2027. When are my instalments due?", ["april", "july"], ["december"]),
    ("C3", "conditional", "I am going into my second year. What percentage must I pay before registration?", ["25%"], []),
    ("C4", "conditional", "I am an international MBBS student. How many instalments can I spread the balance over?", ["9 equal"], []),
    ("C5", "conditional", "I'm Iranian and starting in September 2026. Can I pay in instalments, and when?", ["50%", "december", "march"], ["april"]),
    ("C6", "conditional", "My remaining balance is £60. Can I set up an instalment plan?", ["100"], []),
    ("C7", "conditional", "As a returning student, what percentage is due on 01 March?", ["50%"], []),
    ("C8", "conditional", "I am a January 2027 postgraduate. What is the minimum I pay at the start?", ["50%"], ["25%"]),
    ("A1", "arithmetic", "What is the total yearly cost for an international MBBS student including the levy?", ["51763"], []),
    ("A2", "arithmetic", "I start a standard taught Masters in September 2026 and cannot pay in full. What is the minimum I must pay at registration, in pounds?", ["9155"], []),
    ("A3", "arithmetic", "For a standard taught Masters starting September 2026, how much is each of the two later instalments if I pay the minimum at registration?", ["4577"], []),
    ("A4", "arithmetic", "I am a returning undergraduate. How much must I pay before registration, in pounds?", ["4372"], []),
    ("A5", "arithmetic", "I am a returning undergraduate. How much is due on 01 March, in pounds?", ["8745"], []),
    ("A6", "arithmetic", "For MBBS, what is the minimum deposit in pounds if I use the instalment plan?", ["3963"], []),
]

def norm(t):
    t = (t or "").lower().replace("£", "").replace(",", "").replace("nine", "9")
    t = re.sub(r"(\d)\.(\d{1,2})\b", r"\1", t)      # 4577.50 -> 4577 ; keeps 25% etc.
    return re.sub(r"\s+", " ", t)

def score(answer, keys, forbidden):
    a = norm(answer)
    ok = all(k in a for k in keys)
    bad = any(f in a for f in forbidden)
    return ok and not bad

SYSTEM = "Answer the question using only the policy information below. Be concise and give exact figures."

def make_prompt(rep, q):
    return f"{SYSTEM}\n\nPolicy information:\n{rep}\n\nQuestion: {q}"

def run_pilot(generate):
    rows = []
    for cond, rep in [("text", TEXT_REP), ("table", TABLE_REP)]:
        for qid, cat, q, keys, forb in QUESTIONS:
            ans = generate(make_prompt(rep, q))
            rows.append(dict(condition=cond, id=qid, category=cat, question=q, answer=ans,
                             correct=score(ans, keys, forb)))
    df = pd.DataFrame(rows)
    summary = df.pivot_table(index="category", columns="condition", values="correct", aggfunc="mean")
    summary.loc["OVERALL"] = df.groupby("condition")["correct"].mean()
    # paired comparison
    w = df.pivot(index="id", columns="condition", values="correct")
    table_only = int(((w["table"]) & (~w["text"])).sum())
    text_only = int(((w["text"]) & (~w["table"])).sum())
    return df, summary.round(3), table_only, text_only


# ---------------- run on Mistral-7B-Instruct (bf16, A100) ----------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
if "pilot_model" not in globals():
    pilot_tok = AutoTokenizer.from_pretrained(MODEL_ID)
    pilot_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
    pilot_model.eval()

def generate(prompt, max_new_tokens=160):
    enc = pilot_tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    ids = enc["input_ids"].to(pilot_model.device)
    mask = enc["attention_mask"].to(pilot_model.device)
    with torch.no_grad():
        out = pilot_model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens, do_sample=False,
                                   pad_token_id=pilot_tok.eos_token_id)
    return pilot_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

t0 = time.time()
pilot_df, pilot_summary, table_only, text_only = run_pilot(generate)
print(f"Done in {time.time()-t0:.0f}s, {len(pilot_df)} answers")
print("\n=== PILOT RESULT: share of questions answered correctly ===")
print(pilot_summary.to_string())
print(f"\nPaired: table right & text wrong = {table_only} | text right & table wrong = {text_only}")
from scipy.stats import binomtest
if table_only + text_only:
    print("Exact McNemar p =", round(binomtest(table_only, table_only + text_only, 0.5).pvalue, 4))
print("\n=== Items where the two representations disagree ===")
w = pilot_df.pivot(index="id", columns="condition", values="correct")
for qid in w[w["table"] != w["text"]].index:
    r = pilot_df[pilot_df.id == qid].set_index("condition")
    print(f"[{qid}] {r.loc['text','question']}")
    print("   TEXT :", ("OK " if r.loc['text','correct'] else "X  ") + r.loc['text','answer'][:220].replace("\n"," "))
    print("   TABLE:", ("OK " if r.loc['table','correct'] else "X  ") + r.loc['table','answer'][:220].replace("\n"," "))
pilot_df.to_csv("pilot_table_vs_text_results.csv", index=False)
print("\nSaved pilot_table_vs_text_results.csv")


Done in 97s, 44 answers

=== PILOT RESULT: share of questions answered correctly ===
condition    table   text
category                 
arithmetic   0.833  0.667
conditional  1.000  1.000
lookup       1.000  1.000
OVERALL      0.955  0.909

Paired: table right & text wrong = 1 | text right & table wrong = 0
Exact McNemar p = 1.0

=== Items where the two representations disagree ===
[A3] For a standard taught Masters starting September 2026, how much is each of the two later instalments if I pay the minimum at registration?
   TEXT : X  To find the amount of each of the two later instalments for a standard taught Masters starting September 2026, we first need to determine the total fee and the initial payment at registration.  The total fee for a standa
   TABLE: OK Based on Table 1 and Table 2, the fee for a standard taught Masters starting September 2026 is £18,310. According to Table 2, a new student with a September 2026 start is required to pay 50% of the fee at registration as

S

In [ ]:
wrong = pilot_df[~pilot_df.correct]
for _, r in wrong.iterrows():
    print(f"[{r.id} | {r.condition}] {r.question}")
    print(r.answer)
    print('-'*80)


[A3 | text] For a standard taught Masters starting September 2026, how much is each of the two later instalments if I pay the minimum at registration?
To find the amount of each of the two later instalments for a standard taught Masters starting September 2026, we first need to determine the total fee and the initial payment at registration.

The total fee for a standard taught Masters is £18,310. According to the policy, international students must pay at least 50% of the tuition fee, including the deposit, at registration. Since the deposit is not mentioned for taught Masters, we can assume it is included in the tuition fee. Therefore, the initial payment at registration would be 50% of £18,310 + the deposit.

However, the policy does not provide the exact figure for the deposit. To find the deposit amount, we would need
--------------------------------------------------------------------------------
[A5 | text] I am a returning undergraduate. How much is due on 01 March, in pounds?
